# Dense Retail Shelf Detection — YOLOv8 Training on Google Colab

**CSE471 — Group 10**

## Before you start
1. Click **Runtime → Change runtime type → T4 GPU** (free tier).
2. Have a free [Roboflow](https://roboflow.com) account ready — you'll paste your API key in Cell 3.
3. Optionally connect Google Drive so checkpoints survive a Colab disconnect.

---

## Cell 1 — Verify GPU & install dependencies

In [ ]:
# Verify GPU
!nvidia-smi

# Install required packages
!pip -q install ultralytics roboflow huggingface_hub

import torch
print(f"\nPyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

## Cell 2 — Mount Google Drive (recommended)
Saves checkpoints to Drive so a disconnect does **not** lose progress.

In [ ]:
USE_DRIVE = True   # set False to keep everything in /content (lost on disconnect)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/shelf_yolo_runs'
else:
    PROJECT_DIR = '/content/shelf_yolo_runs'

import os
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Runs saved to: {PROJECT_DIR}")

## Cell 3 — Download dataset from Roboflow

1. Go to [Roboflow Universe](https://universe.roboflow.com) and search for **"retail shelf product detection"**.
2. Pick a dataset (e.g. *Grocery Store Shelf Detection* or *SKU-110K*), click **Download → YOLOv8 format**.
3. Copy your **API key** from Roboflow Settings and paste it below.

> **Recommended datasets:**
> - `grains-shelf/grocery-shelf-detection` — 3 k images, 1 class (product)
> - `retail-shelf-products/retail-shelf-products-detection` — multi-class SKU

In [ ]:
from roboflow import Roboflow

# ── Fill in your Roboflow credentials ──────────────────────────────────────
ROBOFLOW_API_KEY = 'PASTE_YOUR_API_KEY_HERE'
WORKSPACE         = 'PASTE_WORKSPACE_SLUG'    # e.g. 'my-workspace'
PROJECT           = 'PASTE_PROJECT_SLUG'      # e.g. 'retail-shelf-products'
VERSION           = 1                         # dataset version number
# ───────────────────────────────────────────────────────────────────────────

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download('yolov8')
DATA_YAML = f'{dataset.location}/data.yaml'
print(f'Dataset location : {dataset.location}')
print(f'data.yaml path   : {DATA_YAML}')

# Quick sanity check
import yaml, pathlib
cfg = yaml.safe_load(open(DATA_YAML))
print(f'Classes ({cfg["nc"]}) : {cfg["names"]}')

## Cell 4 — (Optional) Use SKU-110K as a fallback / supplement

Skip this cell if your Roboflow dataset is sufficient.
SKU-110K has ~11 k densely-packed shelf images but no class labels (1 class: *product*), which is perfect for a counting / gap detector.

In [ ]:
# ⚠️  Skip this cell if you already have a Roboflow dataset above.

# SKU-110K — download and auto-split into train/val/test with YOLOv8-style labels
# (requires ~8 GB free in /content; uses a Kaggle mirror)

!pip -q install kaggle

# Put your Kaggle API token (kaggle.json) in Drive and copy it here:
# !cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json

# !kaggle datasets download -d thedatasith/sku110k-annotations -p /content/sku110k --unzip

print('Cell skipped — uncomment lines above if needed.')

## Cell 5 — Train YOLOv8

**Hyper-parameter notes for dense retail shelves:**
| Param | Value | Why |
|---|---|---|
| `imgsz` | 960 | larger crops → better small-object recall |
| `batch` | 8 | fits T4 (16 GB) comfortably at 960 px; lower to 4 if OOM |
| `max_det` | 600 | shelves can have hundreds of products |
| `mosaic` | 1.0 | aggressive mosaic augmentation helps dense scenes |
| `close_mosaic` | 10 | disable mosaic in last 10 epochs for stable convergence |
| `iou` | 0.5 | NMS IoU; tune in Cell 7 after training |

In [ ]:
from ultralytics import YOLO

# ── Config ─────────────────────────────────────────────────────────────────
BASE_MODEL = 'yolov8m.pt'   # start with yolov8s.pt for a ~3× faster baseline
RUN_NAME   = 'shelf_v8m'
EPOCHS     = 50
IMGSZ      = 960
BATCH      = 8              # reduce to 4 if you get CUDA out-of-memory
# ───────────────────────────────────────────────────────────────────────────

model = YOLO(BASE_MODEL)

results = model.train(
    data        = DATA_YAML,
    epochs      = EPOCHS,
    imgsz       = IMGSZ,
    batch       = BATCH,
    patience    = 15,          # early stopping
    project     = PROJECT_DIR,
    name        = RUN_NAME,
    exist_ok    = True,        # resume into same folder
    max_det     = 600,
    mosaic      = 1.0,
    scale       = 0.5,
    close_mosaic= 10,
    cache       = True,
    workers     = 2,
    iou         = 0.5,
    verbose     = True,
)

BEST_PT  = f'{PROJECT_DIR}/{RUN_NAME}/weights/best.pt'
LAST_PT  = f'{PROJECT_DIR}/{RUN_NAME}/weights/last.pt'
print(f'\nBest weights : {BEST_PT}')

## Cell 6 — Resume after a Colab disconnect

Re-run Cells 1–2 to reinstall packages and remount Drive, then run this cell.

In [ ]:
# Uncomment and run ONLY if the previous session disconnected mid-training:

# from ultralytics import YOLO
# BEST_PT = f'{PROJECT_DIR}/shelf_v8m/weights/last.pt'
# model = YOLO(BEST_PT)
# model.train(resume=True)

print('Resume cell — uncomment the lines above if needed.')

## Cell 7 — Evaluate at multiple NMS IoU thresholds

On densely packed shelves, the NMS IoU threshold matters a lot.  
Use these numbers in your **CSE471 report**.

In [ ]:
from ultralytics import YOLO

model = YOLO(BEST_PT)
print(f'Evaluating: {BEST_PT}\n')

print(f'{"NMS IoU":<10} {"Precision":<12} {"Recall":<10} {"mAP@50":<10} {"mAP@50-95"}')
print('-' * 55)
for iou_val in (0.45, 0.50, 0.55, 0.60, 0.65, 0.70):
    m = model.val(
        data    = DATA_YAML,
        imgsz   = IMGSZ,
        iou     = iou_val,
        max_det = 600,
        plots   = (iou_val == 0.60),   # save confusion matrix + PR curve once
        verbose = False,
    )
    print(f'{iou_val:<10.2f} {m.box.mp:<12.3f} {m.box.mr:<10.3f} '
          f'{m.box.map50:<10.3f} {m.box.map:.3f}')

## Cell 8 — Visualise training metrics

In [ ]:
from IPython.display import Image as IPImage, display
import pathlib

run_dir = pathlib.Path(f'{PROJECT_DIR}/{RUN_NAME}')

plots = [
    'results.png',
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'PR_curve.png',
    'F1_curve.png',
    'val_batch0_pred.jpg',
    'val_batch1_pred.jpg',
]

for fname in plots:
    p = run_dir / fname
    if p.exists():
        print(f'\n── {fname} ──')
        display(IPImage(str(p), width=900))
    else:
        print(f'[skip] {fname} not found')

## Cell 9 — Run inference on sample images

In [ ]:
import pathlib
from IPython.display import Image as IPImage, display
from ultralytics import YOLO

model = YOLO(BEST_PT)

# Point to a folder of test images (Roboflow test split or your own)
TEST_DIR = pathlib.Path(f'{dataset.location}/test/images')
sample_images = list(TEST_DIR.glob('*.jpg'))[:4]  # first 4 images

for img_path in sample_images:
    preds = model.predict(
        source  = str(img_path),
        conf    = 0.25,
        iou     = 0.60,
        imgsz   = 960,
        max_det = 600,
        save    = True,
        project = '/content/inference_out',
        name    = 'preds',
        exist_ok= True,
    )
    print(f'{img_path.name}: {len(preds[0].boxes)} detections')

# Display saved results
for f in sorted(pathlib.Path('/content/inference_out/preds').glob('*.jpg')):
    display(IPImage(str(f), width=800))

## Cell 10 — Export to ONNX and download weights

**Where to put `best.pt`:**
- Under 100 MB → copy to `models/best.pt` in your GitHub repo.
- Over 100 MB → upload to [Hugging Face Hub](https://huggingface.co/new) and set `HF_REPO_ID` in Streamlit secrets.

In [ ]:
from ultralytics import YOLO
from google.colab import files
import pathlib

model = YOLO(BEST_PT)

# Export ONNX (lighter, faster on CPU-only Streamlit)
onnx_path = model.export(format='onnx', imgsz=IMGSZ, simplify=True, dynamic=False)
print(f'ONNX model: {onnx_path}')

# File sizes
for p in [BEST_PT, onnx_path]:
    size_mb = pathlib.Path(p).stat().st_size / 1e6
    print(f'  {pathlib.Path(p).name}: {size_mb:.1f} MB')

# Download to your local machine
files.download(BEST_PT)
files.download(onnx_path)
files.download(f'{PROJECT_DIR}/{RUN_NAME}/results.csv')
print('\nDownload triggered. Check your browser downloads folder.')

## Cell 11 — (Optional) Push weights to Hugging Face Hub

Use this if `best.pt` is > 100 MB and cannot go into GitHub.

In [ ]:
# Fill in your Hugging Face credentials
HF_TOKEN   = 'PASTE_YOUR_HF_TOKEN'
HF_REPO_ID = 'your-username/shelf-yolov8m'  # will be created if it doesn't exist

from huggingface_hub import HfApi, login
login(token=HF_TOKEN)

api = HfApi()
api.create_repo(HF_REPO_ID, repo_type='model', exist_ok=True)

api.upload_file(
    path_or_fileobj = BEST_PT,
    path_in_repo    = 'best.pt',
    repo_id         = HF_REPO_ID,
    repo_type       = 'model',
)
print(f'Uploaded to https://huggingface.co/{HF_REPO_ID}')
print(f'Set Streamlit secret: HF_REPO_ID = "{HF_REPO_ID}"')

---
## Summary of key files produced

| File | Purpose |
|---|---|
| `best.pt` | Fine-tuned YOLOv8 weights — goes into `models/` in the repo |
| `best.onnx` | ONNX export — lighter, runs on CPU-only Streamlit Cloud |
| `results.csv` | Per-epoch train/val metrics for your report |
| `confusion_matrix.png` | Per-class confusion matrix |
| `PR_curve.png` | Precision-Recall curve |
| `F1_curve.png` | F1 vs confidence curve |

### Next steps
1. Place `best.pt` in `models/` and push to GitHub.
2. Deploy the Streamlit app: **share.streamlit.io → New app → this repo → `app.py`**.
3. If using HF Hub, add `HF_REPO_ID` under **Advanced settings → Secrets**.